# 4.2.2 模型并行  

模型并行（Model Parallelism）往往用于解决单节点内存不足的问题。以包含1750 亿参数的GPT-3 模型为例，如果模型中每一个参数都使用32 位浮点数表示，那么模型需要占用700GB（即$175\mathrm{G}\!\times4$ Bytes）内存，如果使用16 位浮点表示，每个模型副本需要也需要占用350GB 内存。以2022 年3 月NVIDIA 发布的H100 加速卡也仅支持80GB 显存，无法将整个模型完整放入其中。模型并行可以从计算图角度，以下两种形式进行切分：（1）按模型的层切分到不同设备，即层间并行或算子间并行（Inter-operator Parallelism），也称之为流水线并行（Pipeline Parallelism，PP）；（2）将计算图层内的参数切分到不同设备，即层内并行或算子内并行（Intra-operator Parallelism），也称之为张量并行（Tensor Parallelism，TP）。两节点模型并行训练系统样例如图4.5所示，左边为流水线并行，模型的不同层被切分到不同的设备中；右边为张量并行，同一个层中的不同的参数被切分到不同的设备中进行计算。  

# 1. 流水线并行  

流水线并行（Pipeline Parallelism，PP）是一种并行计算策略，将模型的各个层分段处理，并将每个段分布在不同的计算设备上，使得前后阶段能够流水式、分批进行工作。流水线并行通常应用于大规模模型的并行系统中，以有效解决单个计算设备内存不足的问题。图4.6给出了一个由四个计算设备组成的流水线并行系统，包含了前向计算和后向计算。其中F1、F2、F3、F4 分别代表四个前向路径，位于不同的设备上；而B4、B3、B2、B1 则代表逆序的后向路径，也分别位于四个不同的设备上。然而，从图中可以看出，计算图中的下游设备（Downstream Device）需要长时间持续处于空闲状态，等待上游设备（Upstream Device）的计算完成，才能开始计算自身的任务。这种情况导致了设备的平均使用率大幅降低，形成了模型并行气泡（Model Parallelism Bubble），也称为流水线气泡（Pipeline Bubble）。  

![](images/b218f6e3971c244f7371c3f17af7e0bccb1106025de74bc613f9e7c1578b7b3e.jpg)  
图4.5 两节点模型并行训练系统样例  

![](images/a6ee0416c06b8701dd49e712ff54547c62d6f5a2b1b6a801e04d2383f6a3cb79.jpg)  
图4.6 流水线并行样例  

朴素流水线策略所产生的并行气泡，使得系统无法充分利用计算资源，降低了系统整体的计算效率。为了能够减少并行气泡，文献[136] 提出了GPipe 方法，将小批次（Mini-batch）进一步划分成更小的微批次（Micro-batch），利用流水线并行方案，每次处理一个微批次的数据。在当前阶段计算完成得到结果后，将该微批次的结果发送给下游设备，同时开始处理后一个微批次的数据，这样可以在一定程度上减少并行气泡。图4.7给出了GPipe 策略流水线并行样例。如图所示，前向$\mathrm{F_{1}}$ 计算被拆解为了 $\mathrm{F_{11}}$ ， $\mathrm{F_{12}}$ ， $\mathrm{F_{13}}$ ， $\mathrm{F_{14}}$ ，在计算设备1 中计算完成 $\mathrm{F_{11}}$ 后，会在计算设备2 中开始进行 $\mathrm{F_{21}}$ 计算，同时计算设备1 中并行开始 $\mathrm{F_{12}}$ 的计算。相比于最原始的流水线并行方法，GPipe流水线方法可以有效降低并行气泡。  

GPipe 策略虽然可以减少一定的并行气泡，但是只有当一个Mini-batch 中所有的前向计算完成后，才能开始执行后向计算。因此还是会产生很多并行气泡，从而降低了系统的并行效率。Megatron-$\mathrm{LM}^{[137]}$ 提出了1F1B 流水线策略，即一个前向通道和一个后向通道。1F1B 流水线策略引入了任务调度机制，使得下游设备能够在等待上游计算的同时执行其他可并行的任务，从而提高设备的利用率。1F1B 给出了非交错式和交错式两种方式调度方式，如图4.8所示。  

![](images/03085534d0f1999617cff77e1a0e1cc574b5bc883654fd86f3bca5463bf1510f.jpg)  

1F1B 非交错式调度模式可分为三个阶段。首先是热身阶段，在该阶段中，计算设备中进行不同数量的前向计算。接下来的阶段是前向-后向阶段，计算设备按顺序执行一次前向计算，然后进行一次后向计算。最后一个阶段是后向阶段，计算设备再完成最后一次后向计算。相比于GPipe 策略，非交错式调度模式在节省内存方面表现更好。然而，它需要与GPipe 策略一样的时间来完成一轮计算。  

1F1B 交错式调度模式要求micro-batch 的数量是流水线阶段的整数倍。每个设备不再仅负责连续多个层的计算，而是可以处理多个层的子集，这些子集被称为模型块。具体而言，在之前的模式中，设备1 可能负责层1-4，设备2 负责层5-8，以此类推。然而，在新的模式下，设备1 可以处理层1、2、9、10，设备2 处理层3、4、11、12，以此类推。这种模式下，每个设备在流水线中被分配到多个阶段。例如，设备1 可能参与热身阶段、前向计算阶段和后向计算阶段的某些子集任务。每个设备可以并行执行不同阶段的计算任务，从而更好地利用流水线并行的优势。这种模式不仅在内存消耗方面表现出色，还能够提高计算效率，使得大型模型的并行系统能够更高效地完成计算任务。  

![](images/1c15c093a6953795388f5b4b589a80a62c0346ba160ffb223a8f7c26fc84b353.jpg) 

PyTorch 中也包含了实现流水线的API 函数Pipe，具体实现参考“torch.distributed.pipeline.sync.Pipe”类。可以使用这个API 构造一个包含两个线性层，分别放置在2 个不同计算设备中的样例如下： 

In [ ]:
import os
import torch
import torch.nn as nn
from torch.distributed.pipeline.sync import Pipe

# 步骤 0. 需要首先初始化远程过程调用 (RPC) 框架
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'
torch.distributed.rpc.init_rpc('worker', rank=0, world_size=1)
# 步骤 1：构建一个模型，包括两个线性层
fc1 = nn.Linear(16, 8).cuda(0)
fc2 = nn.Linear(8, 4).cuda(1)
# 步骤 2：使用 nn.Sequential 包装这两个层。
model = nn.Sequential(fc1, fc2)
# Step 3: build Pipe (torch.distributed.pipeline.sync.Pipe)
model = Pipe(model, chunks=8)
input = torch.rand(16, 16).cuda(0)
output_rref = model(input)

## 2. 张量并行  

张量并行（Tensor Parallelism，TP）需要根据模型的具体结构和算子类型，解决如何将参数切分到不同设备，以及如何保证切分后数学一致性两个问题。大语言模型都是以Transformer 结构为基础，Transformer 结构主要由以下三种算子构成：嵌入式表示（Embedding）、矩阵乘（MatMul）和交叉熵损失（Cross Entropy Loss）计算构成。这三种类型的算子有较大的差异，都需要设计对应的张量并行策略[135]，才可以实现将参数切分到不同的设备。  

对于嵌入表示（Embedding）算子，如果总的词表数非常大，会导致单计算设备显存无法容纳Embedding 层参数。举例来说，如果词表数量是64000，嵌入表示维度为5120，类型采用32 位精度浮点数，那么整层参数需要的显存大约为 $64000\times5120\times4/1024/1024=1250\mathrm{MB},$ ，反向梯度同样需要1250MB，仅仅存储就需要将近 $2.5\mathrm{GB}.$ 。对于嵌入表示层的参数，可以按照词维度切分，每个计算设备只存储部分词向量，然后通过汇总各个设备上的部分词向量，从而得到完整的词向量。图4.9给出了单节点Embedding 和两节点张量并行的示意图。在单节点上，执行Embedding 操作，bz 是批次大小（batch size），Embedding 的参数大小为[word_size, hidden_size]，计算得到[bz,hidden_size] 张量。图4.9中Embedding 张量并行示例将Embedding 参数沿word_size 维度，切分为两块，每块大小为[word_size/2, hidden_size]，分别存储在两个设备上。当每个节点查询各自的词表时，如果无法查到，则该词的表示为0，各自设备查询后得到[bz, hidden_size] 结果张量，最后通过AllReduce_Sum 通信x，跨设备求和，得到完整的全量结果，可以看出，这里的输出结果和单计算设备执行的结果一致。 


![](images/aa736d5e4ca1b5b84562f40f511e8e5c2c4bea15ab3c091c6dd2295328222f62.jpg) 

(1) 参数矩阵 $\pmb{A}$ 按列切块，将矩阵 $\pmb{A}$ 按列切成：  

$$
\pmb{A}=[A_{1},A_{2}]
$$  

(2) 参数矩阵 $\pmb{A}$ 按行切块，将矩阵 $\pmb{A}$ 按行切成：  

$$
A={\binom{A_{1}}{A_{2}}}
$$  

图4.10给出了参数矩阵按列切分的示例，参数矩阵 $\pmb{A}$ 分别将 $A_{1}$ ， $A_{2}$ 放置在两个计算设备上。  

两个计算设备分别计算 $Y_{1}=X\times A_{1}$ 和 $Y_{2}=X\times A_{2}$ 。计算完成后，多计算设备间进行通信，从而获取其它计算设备上的计算结果，并拼接在一起得到最终的结果矩阵 $\mathbf{\deltaY}$ ，该结果在数学上与单计算设备计算结果上完全等价。 

![](images/f3b99cb49f1a9cf28989e6a4a9c420643f72929e8f8c42064db51da8ebc54b03.jpg)  

图4.11给出了参数矩阵按列行分的示例，为了满足矩阵乘法规则，输入矩阵 $\mathbf{\deltaX}$ 需要按列切分$\pmb{X}=[X_{1}|X_{2}]_{\mathfrak{c}}$ 。同时，将矩阵分块，分别放置在两个计算设备上，每个计算设备分别计算 ${\cal Y}_{1}=$ $X_{1}\times A_{1}$ 和 $Y_{2}=X_{2}\times A_{2}$ 。计算完成后，多个计算设备间通信获取归约其他卡上的计算结果，可以得到最终的结果矩阵 $\mathbf{\deltaY}$ 。同样，这种切分方式，既可以保证数学上的计算等价性，并解决单计算设备显存无法容纳，又可以保证单计算设备通过拆分方式可以装下参数 $\pmb{A}$ 的问题。  

Transformer 中的FFN 结构均包含两层全连接（FC）层，即存在两个矩阵乘，这两个矩阵乘分别采用上述两种切分方式，如图4.12所示。对第一个FC 层的参数矩阵按列切块，对第二个FC层参数矩阵按行切块。这样第一个FC 层的输出恰好满足第二个FC 层数据输入要求（按列切分），因此可以省去第一个FC 层后的汇总通信操作。多头自注意力机制的张量并行与FFN 类似，因为具有多个独立的头，因此相较于FFN 更容易实现并行，其矩阵切分方式如图4.13所示。具体可以参考文献[135]。  

分类网络最后一层一般会选用Softmax 和Cross_entropy 算子来计算交叉熵损失（Cross EntropyLoss）。如果类别数量非常大，会导致单计算设备内存无法存储和计算logit 矩阵。针对这一类算子，可以按照类别维度切分，同时通过中间结果通信，得到最终的全局的交叉熵损失。首先计算的是softmax 值，公式如下：  

$$
\operatorname{Softmax}(x_{i})={\frac{e^{x_{i}}}{\sum_{j}e^{x_{j}}}}={\frac{e^{x_{i}-x_{m a x}}}{\sum_{j}e^{x_{j}-x_{m a x}}}}={\frac{e^{x_{i}-x_{m a x}}}{\sum_{N}\!\sum_{j}e^{x_{j}-x_{m a x}}}}
$$  

$$
x_{m a x}=\operatorname*{max}_{p}(\operatorname*{max}_{k}(x_{k}))
$$   
其中， $p$ 表示张量并行的设备号。得到Softmax 计算结果之后，同时对标签Target 按类别切分，每个设备得到部分损失，最后再进行一次通信，得到所有类别的损失。整个过程，只需要进行三次小量的通信，就可以完成交叉熵损失的计算。 


![](images/f4af9fc9bc1e4547ea1c397ba95a5725cd5c899fa91e4b4177fccc7300d77982.jpg)  
图4.11 两节点矩阵乘算子张量并行按行切分示例  

![](images/635dc7a406694278d721c56dd1cdb1a8f4a2a6c357c876ab818d7444f5d401d1.jpg)  
图4.12 FNN 结构张量并行示意图[135]  

PyTorch 提供了细粒度张量级别的并行API，DistributedTensor。也提供了粗粒度模型层面的API 对“nn.Module”进行张量并行。通过以下几行代码就可以实现对一个大的张量进行分片：

In [ ]:
import torch
from torch.distributed._tensor import DTensor, DeviceMesh, Shard, distribute_tensor
# 使用可用设备构建设备网格（多主机或单主机）
device_mesh = DeviceMesh("cuda", [0, 1, 2, 3])
# 如果想要进行逐行分片
rowwise_placement=[Shard(0)]
# 如果想要进行逐列分片
colwise_placement=[Shard(1)]
big_tensor = torch.randn(888, 12)
# 分布式张量返回将根据指定的放置维度进行分片
rowwise_tensor = distribute_tensor(big_tensor, device_mesh=device_mesh, placements=rowwise_placement)

对于像“nn.Linear”这样已经有“torch.Tensor”作为参数的模块，也提供了模块级API“dis
tribute_module”在模型层面进行张量并行，参考代码如下：

In [ ]:
import torch
from torch.distributed._tensor import DeviceMesh, Shard, distribute_tensor,distribute_module
class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 8)
        self.fc2 = nn.Linear(8, 8)
        self.relu = nn.ReLU()
        
    def forward(self, input):
        return self.relu(self.fc1(input) + self.fc2(input))
    
mesh = DeviceMesh(device_type="cuda", mesh=[[0, 1], [2, 3]])

def shard_params(mod_name, mod, mesh):
    rowwise_placement = [Shard(0)]
    def to_dist_tensor(t): return distribute_tensor(t, mesh, rowwise_placement)
    mod._apply(to_dist_tensor)
    
sharded_module = distribute_module(MyModule(), mesh, partition_fn=shard_params)
    
def shard_fc(mod_name, mod, mesh):
    rowwise_placement = [Shard(0)]
    if mod_name == "fc1":
        mod.weight = torch.nn.Parameter(distribute_tensor(mod.weight, mesh, rowwise_placement))

sharded_module = distribute_module(MyModule(), mesh, partition_fn=shard_fc)